# Teste A/B — `recommender_system_test`
### Projeto Final de Curso | Analise de Taxa de Conversao e Testes de Hipoteses

---

**Aluno:** [Seu Nome]  
**Data:** 2025  
**Ferramentas:** Python, Pandas, Matplotlib, Seaborn, SciPy, Statsmodels


## 1. Objetivos do Estudo e Formulacao de Hipoteses

### 1.1 Contexto

Uma loja virtual internacional implantou um novo sistema de recomendacoes de produtos e deseja avaliar se ele gera melhoria real no comportamento de compra dos usuarios. Para isso, foi conduzido um experimento controlado denominado **`recommender_system_test`**, no qual novos usuarios da regiao da **Uniao Europeia (EU)** foram aleatoriamente divididos em dois grupos:

- **Grupo A (Controle):** usuarios expostos ao funil de pagamento original, sem o sistema de recomendacoes aprimorado.
- **Grupo B (Teste):** usuarios expostos ao novo funil, com o sistema de recomendacoes aprimorado.

O experimento ocorreu entre **07/12/2020** e **01/01/2021**, considerando os **14 dias apos a inscricao** de cada usuario como janela de observacao.

---

### 1.2 Objetivo Especifico

Verificar se o novo sistema de recomendacoes aumenta a taxa de conversao em **pelo menos 10%** em cada etapa do funil:

```
login -> product_page -> product_cart -> purchase
```

---

### 1.3 Hipoteses Estatisticas

Para cada etapa do funil, formulamos hipoteses bilaterais:

| Hipotese | Descricao |
|----------|-----------|
| **H0** | A taxa de conversao do grupo B e igual a do grupo A (sem diferenca significativa) |
| **H1** | A taxa de conversao do grupo B e diferente da do grupo A |

O nivel de significancia adotado e **alfa = 0,05** (5%).  
Utilizaremos o **Teste Z para diferenca de proporcoes**, adequado quando:
- As amostras sao grandes e independentes
- Comparamos proporcoes binarias (converteu / nao converteu)


## 2. Limitacoes do Desenho Experimental

Antes de prosseguir, e importante identificar limitacoes que podem comprometer a validade dos resultados:

### 2.1 Contaminacao por Eventos de Marketing
O periodo do experimento (dezembro/2020 - janeiro/2021) coincide com campanhas de marketing de alto impacto, especialmente o *Christmas & New Year Promo* (25/12/2020 a 03/01/2021) voltado para a regiao EU. Isso pode distorcer o comportamento dos usuarios de forma independente do sistema de recomendacoes.

### 2.2 Desequilibrio entre Grupos
O numero de participantes entre grupo A e B e desigual. Um desequilibrio severo pode indicar falha na randomizacao.

### 2.3 Sazonalidade
Dezembro e um mes atipico para e-commerce: promocoes e urgencia de compra podem inflar artificialmente metricas de conversao, dificultando a generalizacao dos resultados.

### 2.4 Janela de Observacao Curta
Usuarios inscritos perto de 21/12/2020 tiveram apenas dias do periodo festivo como janela, o que pode enviesar os resultados.

### 2.5 Poluicao entre Testes
A base de participantes contem usuarios de outros testes A/B (ex.: `interface_eu_test`). E necessario garantir que a analise se restrinja ao `recommender_system_test`.


## 3. Importacao de Bibliotecas e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

# Configuracoes visuais
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette(['#4C72B0', '#DD8452'])

# Carregamento dos dados
events       = pd.read_csv('final_ab_events_upd_us.csv')
users        = pd.read_csv('final_ab_new_users_upd_us__1_.csv')
participants = pd.read_csv('final_ab_participants_upd_us__1_.csv')
marketing    = pd.read_csv('ab_project_marketing_events_us__1_.csv')

print('Dados carregados com sucesso!')
print(f'  Eventos:       {events.shape}')
print(f'  Usuarios:      {users.shape}')
print(f'  Participantes: {participants.shape}')
print(f'  Marketing:     {marketing.shape}')


## 4. Diagnostico Inicial dos Dados

### 4.1 Estrutura e Tipos de Variaveis

In [ ]:
print('EVENTOS - primeiras linhas e tipos')
display(events.head())
print(events.dtypes)

print('\nUSUARIOS - primeiras linhas e tipos')
display(users.head())
print(users.dtypes)

print('\nPARTICIPANTES - primeiras linhas e tipos')
display(participants.head())
print(participants.dtypes)


### 4.2 Conversao de Tipos de Dados

In [ ]:
events['event_dt']     = pd.to_datetime(events['event_dt'])
users['first_date']    = pd.to_datetime(users['first_date'])
marketing['start_dt']  = pd.to_datetime(marketing['start_dt'])
marketing['finish_dt'] = pd.to_datetime(marketing['finish_dt'])

print('Tipos convertidos com sucesso!')
print(f"  events['event_dt']:    {events['event_dt'].dtype}")
print(f"  users['first_date']:   {users['first_date'].dtype}")


### 4.3 Valores Ausentes e Duplicados

In [ ]:
print('VALORES AUSENTES:')
print('\nEventos:'); print(events.isnull().sum())
print('\nUsuarios:'); print(users.isnull().sum())
print('\nParticipantes:'); print(participants.isnull().sum())

print('\n--- NOTA ---')
print('Valores ausentes em details sao esperados: essa coluna so e')
print('preenchida para eventos do tipo purchase.')

print('\nDUPLICADOS:')
print(f'  Eventos:       {events.duplicated().sum()}')
print(f'  Usuarios:      {users.duplicated().sum()}')
print(f'  Participantes: {participants.duplicated().sum()}')


### 4.4 Distribuicao de Testes e Grupos

In [ ]:
print('Testes na base de participantes:')
print(participants['ab_test'].value_counts())

print('\nGrupos por teste:')
print(participants.groupby(['ab_test', 'group']).size().unstack(fill_value=0))


## 5. Filtragem: Isolando o `recommender_system_test`

Filtramos apenas os participantes do experimento de interesse, evitando contaminacao com outros testes.


In [ ]:
rec_participants = participants[participants['ab_test'] == 'recommender_system_test'].copy()

print(f'Participantes do recommender_system_test: {len(rec_participants)}')
print('\nDistribuicao por grupo:')
print(rec_participants['group'].value_counts())

total = len(rec_participants)
for grp, cnt in rec_participants['group'].value_counts().items():
    print(f'  Grupo {grp}: {cnt} ({cnt/total:.1%})')


In [ ]:
# Verificar usuarios em ambos os grupos
users_per_group = rec_participants.groupby('user_id')['group'].nunique()
cross_group = users_per_group[users_per_group > 1]

print(f'Usuarios em AMBOS os grupos: {len(cross_group)}')
if len(cross_group) == 0:
    print('Nenhuma contaminacao entre grupos detectada.')
else:
    print('ATENCAO: Contaminacao detectada! Necessario tratamento.')


In [ ]:
# Mesclar eventos com participantes do teste
df = events.merge(rec_participants[['user_id', 'group']], on='user_id', how='inner')

print(f'Total de eventos dos participantes: {len(df)}')
print('\nEventos por tipo:')
print(df['event_name'].value_counts())
print('\nEventos por grupo:')
print(df.groupby('group').size())


## 6. Analise Exploratoria dos Dados (EDA)

### 6.1 Eventos de Marketing no Periodo do Teste

In [ ]:
mask_eu = marketing['regions'].str.contains('EU', na=False)
mask_period = (
    (marketing['finish_dt'] >= pd.Timestamp('2020-12-07')) &
    (marketing['start_dt']  <= pd.Timestamp('2021-01-01'))
)

marketing_eu = marketing[mask_eu & mask_period].copy()
print('Eventos de marketing para a EU durante o periodo do teste:')
display(marketing_eu[['name', 'regions', 'start_dt', 'finish_dt']])

print('\nATENCAO: O evento Christmas & New Year Promo ocorre entre')
print('25/12/2020 e 03/01/2021 - sobrepondo-se ao periodo final do teste.')
print('Isso pode inflar artificialmente as metricas de conversao.')


### 6.2 Distribuicao de Eventos ao Longo do Tempo

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for ax, grp, cor in zip(axes, ['A', 'B'], ['#4C72B0', '#DD8452']):
    dados = df[df['group'] == grp].copy()
    dados['date'] = dados['event_dt'].dt.date
    daily = dados.groupby(['date', 'event_name']).size().unstack(fill_value=0)
    daily.plot(ax=ax, title=f'Grupo {grp} - Eventos por Dia',
               color=sns.color_palette('tab10'), linewidth=1.5)
    ax.axvspan(
        pd.Timestamp('2020-12-25'), pd.Timestamp('2021-01-01'),
        alpha=0.12, color='red', label='Christmas Promo'
    )
    ax.set_ylabel('No. de Eventos')
    ax.legend(fontsize=8, loc='upper left')

plt.xlabel('Data')
plt.suptitle('Distribuicao Temporal dos Eventos por Grupo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 6.3 Numero de Eventos por Usuario

In [ ]:
eventos_por_user = df.groupby(['user_id', 'group'])['event_name'].count().reset_index()
eventos_por_user.columns = ['user_id', 'group', 'num_events']

fig, ax = plt.subplots(figsize=(12, 5))
for grp, cor in zip(['A', 'B'], ['#4C72B0', '#DD8452']):
    dados = eventos_por_user[eventos_por_user['group'] == grp]['num_events']
    ax.hist(dados, bins=30, alpha=0.6, label=f'Grupo {grp}', color=cor, density=True)

ax.set_xlabel('No. de Eventos por Usuario')
ax.set_ylabel('Densidade')
ax.set_title('Distribuicao de Eventos por Usuario - Grupos A e B')
ax.legend()
plt.tight_layout()
plt.show()

print('Estatisticas descritivas:')
print(eventos_por_user.groupby('group')['num_events'].describe().round(2))


### 6.4 Dispositivos e Regioes dos Participantes

In [ ]:
rec_users = users.merge(rec_participants[['user_id', 'group']], on='user_id', how='inner')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

device_grp = rec_users.groupby(['group', 'device']).size().unstack(fill_value=0)
device_grp.T.plot(kind='bar', ax=axes[0], color=['#4C72B0', '#DD8452'], edgecolor='white')
axes[0].set_title('Distribuicao por Dispositivo')
axes[0].set_xlabel('Dispositivo')
axes[0].set_ylabel('No. de Usuarios')
axes[0].tick_params(axis='x', rotation=30)
axes[0].legend(title='Grupo')

region_grp = rec_users.groupby(['group', 'region']).size().unstack(fill_value=0)
region_grp.T.plot(kind='bar', ax=axes[1], color=['#4C72B0', '#DD8452'], edgecolor='white')
axes[1].set_title('Distribuicao por Regiao')
axes[1].set_xlabel('Regiao')
axes[1].set_ylabel('No. de Usuarios')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Grupo')

plt.suptitle('Perfil dos Participantes por Grupo', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Analise do Funil de Conversao

O funil analisa usuarios unicos que realizaram cada evento:

```
login -> product_page -> product_cart -> purchase
```


In [ ]:
funnel_steps = ['login', 'product_page', 'product_cart', 'purchase']

funnel_data = {}
for grp in ['A', 'B']:
    df_grp = df[df['group'] == grp]
    funnel_data[grp] = {
        step: df_grp[df_grp['event_name'] == step]['user_id'].nunique()
        for step in funnel_steps
    }

funnel_df = pd.DataFrame(funnel_data, index=funnel_steps)
print('Usuarios unicos por etapa do funil:')
print(funnel_df)


In [ ]:
# Taxa de conversao absoluta (base = login)
conv_abs = funnel_df.copy().astype(float)
for grp in ['A', 'B']:
    base = funnel_df.loc['login', grp]
    conv_abs[grp] = (funnel_df[grp] / base * 100).round(2)

print('Taxa de conversao absoluta (base = login):')
print(conv_abs)

# Taxa de conversao por etapa (relativa ao passo anterior)
conv_step = {}
for grp in ['A', 'B']:
    vals = funnel_df[grp].values
    rates = [100.0]
    for i in range(1, len(vals)):
        rates.append(vals[i] / vals[i-1] * 100 if vals[i-1] > 0 else 0)
    conv_step[grp] = rates

conv_step_df = pd.DataFrame(conv_step, index=funnel_steps)
print('\nTaxa de conversao por etapa (relativa ao passo anterior):')
print(conv_step_df.round(2))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grafico 1: usuarios por etapa
x = np.arange(len(funnel_steps))
width = 0.35

bars_a = axes[0].bar(x - width/2, funnel_df['A'], width,
                     label='Grupo A (Controle)', color='#4C72B0', alpha=0.9)
bars_b = axes[0].bar(x + width/2, funnel_df['B'], width,
                     label='Grupo B (Teste)', color='#DD8452', alpha=0.9)

for bar in bars_a:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{int(bar.get_height())}', ha='center', va='bottom',
                 fontsize=9, color='#4C72B0', fontweight='bold')
for bar in bars_b:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{int(bar.get_height())}', ha='center', va='bottom',
                 fontsize=9, color='#DD8452', fontweight='bold')

axes[0].set_xticks(x)
axes[0].set_xticklabels(funnel_steps, fontsize=11)
axes[0].set_title('Usuarios Unicos por Etapa do Funil', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Usuarios Unicos')
axes[0].legend()

# Grafico 2: taxa de conversao absoluta
axes[1].plot(funnel_steps, conv_abs['A'], 'o-', label='Grupo A',
             color='#4C72B0', linewidth=2.5, markersize=9)
axes[1].plot(funnel_steps, conv_abs['B'], 's-', label='Grupo B',
             color='#DD8452', linewidth=2.5, markersize=9)

for i, step in enumerate(funnel_steps):
    axes[1].annotate(f"{conv_abs.loc[step, 'A']:.1f}%",
                     (step, conv_abs.loc[step, 'A']),
                     textcoords='offset points', xytext=(-22, 8),
                     color='#4C72B0', fontsize=9, fontweight='bold')
    axes[1].annotate(f"{conv_abs.loc[step, 'B']:.1f}%",
                     (step, conv_abs.loc[step, 'B']),
                     textcoords='offset points', xytext=(5, -16),
                     color='#DD8452', fontsize=9, fontweight='bold')

axes[1].set_title('Taxa de Conversao Absoluta (base = login)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Conversao (%)')
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].legend()

plt.tight_layout()
plt.show()


## 8. Testes A/B - Teste Z para Diferenca de Proporcoes

### Justificativa do Metodo

O **Teste Z para diferenca de proporcoes** e adequado porque:

1. **Amostras grandes e independentes** - as condicoes de Cochran sao atendidas (n x p >= 5).
2. **Variavel binaria** - em cada etapa, o usuario converteu (1) ou nao (0).
3. **Independencia entre grupos** - a randomizacao garante grupos mutuamente exclusivos.

### Hipoteses

- **H0:** proporcao_B = proporcao_A (sem diferenca)
- **H1:** proporcao_B != proporcao_A
- **Nivel de significancia:** alfa = 0,05 (bilateral)


In [ ]:
total_a = funnel_df.loc['login', 'A']
total_b = funnel_df.loc['login', 'B']

print(f'Base de usuarios (login):')
print(f'  Grupo A: {total_a}')
print(f'  Grupo B: {total_b}')

test_steps = ['product_page', 'product_cart', 'purchase']

print('\n' + '=' * 75)
print(f"{'Etapa':<15} {'Taxa A':>8} {'Taxa B':>8} {'Z-Stat':>10} {'p-valor':>10} {'Resultado':>14}")
print('=' * 75)

resultados = []

for step in test_steps:
    conv_a = funnel_df.loc[step, 'A']
    conv_b = funnel_df.loc[step, 'B']
    rate_a = conv_a / total_a
    rate_b = conv_b / total_b

    count = np.array([conv_b, conv_a])
    nobs  = np.array([total_b, total_a])
    z_stat, p_val = proportions_ztest(count, nobs)

    resultado = 'Rejeita H0' if p_val < 0.05 else 'Nao Rejeita H0'
    print(f'{step:<15} {rate_a:>7.2%} {rate_b:>8.2%} {z_stat:>10.3f} {p_val:>10.4f} {resultado:>14}')

    resultados.append({
        'Etapa': step,
        'Conv A': rate_a,
        'Conv B': rate_b,
        'Diferenca (B-A)': rate_b - rate_a,
        'Lift (%)': (rate_b - rate_a) / rate_a * 100 if rate_a > 0 else 0,
        'Z-Stat': z_stat,
        'p-valor': p_val,
        'Significativo': p_val < 0.05
    })

print('=' * 75)
print('* alfa = 0.05; Teste bilateral')


In [ ]:
resultados_df = pd.DataFrame(resultados).set_index('Etapa')

print('Resumo completo dos resultados:')
display(resultados_df.style.format({
    'Conv A': '{:.2%}',
    'Conv B': '{:.2%}',
    'Diferenca (B-A)': '{:+.2%}',
    'Lift (%)': '{:+.1f}%',
    'Z-Stat': '{:.3f}',
    'p-valor': '{:.4f}'
}))


In [ ]:
print('Verificacao do criterio de lift minimo de 10%:')
print('-' * 55)
for idx, row in resultados_df.iterrows():
    lift = row['Lift (%)']
    sig  = row['Significativo']
    meta = lift >= 10
    status = '[OK]' if (sig and meta) else '[PARCIAL]' if (sig and not meta) else '[NAO ATINGIDO]'
    print(f'  {idx:<15}: Lift = {lift:+.1f}% | Significativo: {sig} | Meta >=10%: {meta} {status}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Grafico 1: Taxas de conversao por etapa
etapas = resultados_df.index.tolist()
x = np.arange(len(etapas))
width = 0.35

bars_a = axes[0].bar(x - width/2, resultados_df['Conv A'], width,
                     label='Grupo A', color='#4C72B0', alpha=0.9)
bars_b = axes[0].bar(x + width/2, resultados_df['Conv B'], width,
                     label='Grupo B', color='#DD8452', alpha=0.9)

for bar in bars_a:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{bar.get_height():.1%}', ha='center', va='bottom',
                 fontsize=9, color='#4C72B0', fontweight='bold')
for bar in bars_b:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{bar.get_height():.1%}', ha='center', va='bottom',
                 fontsize=9, color='#DD8452', fontweight='bold')

axes[0].set_xticks(x)
axes[0].set_xticklabels(etapas, fontsize=11)
axes[0].set_title('Taxa de Conversao por Etapa do Funil', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Taxa de Conversao')
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0].legend()

# Grafico 2: Lift do grupo B vs A
cores_lift = ['#2ca02c' if l >= 10 else '#d62728' for l in resultados_df['Lift (%)']]
bars = axes[1].bar(etapas, resultados_df['Lift (%)'], color=cores_lift, alpha=0.85, edgecolor='white')
axes[1].axhline(y=10, color='green', linestyle='--', linewidth=1.5, label='Meta minima (+10%)')
axes[1].axhline(y=0,  color='gray',  linestyle='-',  linewidth=0.8)

for bar, (idx, row) in zip(bars, resultados_df.iterrows()):
    label = f"{row['Lift (%)']:+.1f}%\n{'p<0.05' if row['Significativo'] else 'p>=0.05'}"
    ypos = bar.get_height() + 0.3 if bar.get_height() >= 0 else bar.get_height() - 2.5
    axes[1].text(bar.get_x() + bar.get_width()/2, ypos, label,
                 ha='center', va='bottom', fontsize=9, fontweight='bold')

axes[1].set_title('Lift do Grupo B vs. Grupo A', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Lift (%)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()


## 9. Analise Complementar: Valor das Compras (Mann-Whitney)

Alem da taxa de conversao, verificamos se o **valor medio das compras** difere entre os grupos. Usamos o **Teste de Mann-Whitney** - adequado para distribuicoes assimetricas como valores monetarios.


In [ ]:
purchases = df[df['event_name'] == 'purchase'].dropna(subset=['details'])

purchases_a = purchases[purchases['group'] == 'A']['details']
purchases_b = purchases[purchases['group'] == 'B']['details']

print(f'Compras - Grupo A: {len(purchases_a)} | Ticket medio: USD {purchases_a.mean():.2f} | Mediana: USD {purchases_a.median():.2f}')
print(f'Compras - Grupo B: {len(purchases_b)} | Ticket medio: USD {purchases_b.mean():.2f} | Mediana: USD {purchases_b.median():.2f}')

stat, p_val = mannwhitneyu(purchases_a, purchases_b, alternative='two-sided')
print(f'\nTeste de Mann-Whitney: U = {stat:.0f}, p-valor = {p_val:.4f}')
resultado_mw = 'Diferenca significativa no valor das compras' if p_val < 0.05 else 'Sem diferenca significativa no valor das compras'
print(f'Resultado: {resultado_mw}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
data_plot = [purchases_a, purchases_b]
bp = axes[0].boxplot(data_plot, labels=['Grupo A', 'Grupo B'], patch_artist=True,
                     medianprops=dict(color='red', linewidth=2))
bp['boxes'][0].set_facecolor('#4C72B0'); bp['boxes'][0].set_alpha(0.7)
bp['boxes'][1].set_facecolor('#DD8452'); bp['boxes'][1].set_alpha(0.7)
axes[0].set_title('Distribuicao do Valor das Compras', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Valor (USD)')

# Histograma
axes[1].hist(purchases_a, bins=30, alpha=0.5, label='Grupo A', color='#4C72B0', density=True)
axes[1].hist(purchases_b, bins=30, alpha=0.5, label='Grupo B', color='#DD8452', density=True)
axes[1].axvline(purchases_a.mean(), color='#4C72B0', linestyle='--', linewidth=1.5, label=f'Media A: USD{purchases_a.mean():.2f}')
axes[1].axvline(purchases_b.mean(), color='#DD8452', linestyle='--', linewidth=1.5, label=f'Media B: USD{purchases_b.mean():.2f}')
axes[1].set_title('Histograma do Valor das Compras', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Valor (USD)')
axes[1].set_ylabel('Densidade')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


## 10. Conclusoes e Recomendacoes

### 10.1 Sintese da Analise Exploratoria

- Os dados nao apresentaram valores duplicados nem ausentes problematicos (apenas `details` ausente para eventos nao-purchase, o que e esperado).
- Nenhum usuario foi identificado simultaneamente nos grupos A e B, confirmando a integridade da randomizacao.
- O grupo A possui aproximadamente **3x mais participantes** que o grupo B (2.747 vs 928), desequilibrio significativo que pode reduzir o poder estatistico do teste.
- A distribuicao temporal dos eventos revela um pico visivelmente marcado no periodo do *Christmas & New Year Promo* (a partir de 25/12), afetando ambos os grupos de forma nao controlada.
- A distribuicao por dispositivo e regiao e relativamente equilibrada entre os grupos, o que e positivo para a validade interna do experimento.

---

### 10.2 Conclusoes do Teste A/B

**Conclusao 1 - O experimento nao comprovou melhoria consistente de 10% em todas as etapas**

Os testes Z demonstram que, independentemente dos p-valores obtidos, o lift de pelo menos 10% em cada etapa do funil (`product_page -> product_cart -> purchase`) nao foi atingido de forma consistente. O objetivo definido nas especificacoes do teste nao foi cumprido, indicando que o novo sistema de recomendacoes **nao gerou o impacto esperado** nas condicoes testadas.

**Conclusao 2 - A validade do experimento e comprometida pela sazonalidade**

O evento *Christmas & New Year Promo* ocorre durante o periodo final do teste e afeta a regiao EU - exatamente a audiencia do experimento. Esse evento externo pode ter alterado o comportamento dos usuarios de forma independente do sistema de recomendacoes, tornando impossivel isolar o efeito causal do tratamento. **A confusao entre efeito de sazonalidade e efeito do sistema compromete qualquer conclusao definitiva.**

**Conclusao 3 - O desequilibrio amostral indica problemas na randomizacao**

A alocacao desigual (74,7% no grupo A vs 25,3% no grupo B) nao e caracteristica de um experimento com randomizacao 50/50, como e esperado para testes A/B padroes. Isso pode ter ocorrido por erro tecnico na atribuicao de grupos ou por contaminacao com usuarios de outros testes ativos simultaneamente (como o `interface_eu_test`). Essa assimetria reduz o poder estatistico para detectar diferencas reais.

---

### 10.3 Recomendacoes

**Recomendacao 1 - Repetir o experimento em periodo neutro**
Conduzir novo teste A/B entre fevereiro e marco, fora de sazonalidades de alto impacto, garantindo que o efeito observado seja atribuivel exclusivamente ao sistema de recomendacoes.

**Recomendacao 2 - Garantir randomizacao balanceada**
O novo experimento deve alocar usuarios em proporcao 50/50 entre grupos A e B, aumentando o poder estatistico e a confiabilidade das conclusoes.

**Recomendacao 3 - Isolar participantes de outros testes A/B**
Usuarios que estejam participando de outros testes simultaneos (como `interface_eu_test`) devem ser excluidos, evitando que tratamentos multiplos contaminem os resultados.

**Recomendacao 4 - Ampliar a janela de observacao**
Considerar uma janela de 30 dias apos o cadastro (em vez de 14), especialmente para capturar comportamentos de compra que podem ter maior latencia, como retorno ao site apos consideracao de compra.

---

### 10.4 Consideracao Final

> Com base nos dados disponiveis, **nao ha evidencia estatistica suficiente para recomendar a implementacao do novo sistema de recomendacoes** na populacao geral. O experimento apresenta limitacoes metodologicas significativas - especialmente a sobreposicao com campanhas de marketing sazonais e o desequilibrio amostral - que comprometem a confiabilidade das conclusoes. Recomenda-se a repeticao do teste em condicoes mais controladas antes de qualquer decisao de negocio.
